# Project: Personal Chef Agent

## Introduction

This notebook builds a small recipe assistant with LangChain and LangGraph. The agent accepts leftover ingredients, searches the web for recipe ideas, and returns suggestions through a conversational interface.

The example demonstrates four reusable patterns:

- Loading API credentials from a local `.env` file.
- Wrapping an external capability, Tavily web search, as a LangChain tool.
- Creating an agent with a system prompt and a tool.
- Adding short-term conversation memory with a checkpointer and thread ID.

### Prerequisites

Before running the notebook, install the repository dependencies and create a `.env` file containing `OPENAI_API_KEY` and `TAVILY_API_KEY`. The model name in the agent cell must also be available through the configured provider.

In [10]:
# Read key-value pairs from .env into the process environment.
# This keeps API keys out of the notebook source and makes the example portable.
from dotenv import load_dotenv

load_dotenv()

True

In [11]:
# `tool` turns a normal Python function into a tool the agent can call.
from langchain.tools import tool
from typing import Any, Dict
from tavily import TavilyClient

In [12]:
# The client reads TAVILY_API_KEY from the environment loaded above.
tavily_client = TavilyClient()


@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for recipe and ingredient information.

    The type annotation and docstring become part of the tool schema and
    help the agent decide when and how to call this function.
    """
    # Tavily returns a dictionary containing search results and metadata.
    return tavily_client.search(query=query, max_results=5)

### Summary: Tool Definition

`web_search` is the agent's only external capability. The `@tool` decorator exposes the function name, description, and `query` argument to the model. Keeping the function small makes it easy to replace Tavily with another search provider later.

In [13]:
# The system prompt defines the agent's role and the behavior expected
# whenever it receives a user's ingredient list.
system_prompt = """
You are a personal chef. The user will give you a list of
ingredients they have left over in their house. Using the
web search tool, search the web for recipes that can be
made with the ingredients they have. Return recipe suggestions
and eventually the recipe instructions to the user, if requested.
"""

### Summary: Agent Instructions

The system prompt separates stable behavior from user input. It tells the model when to use the search tool and establishes the expected output, while the actual ingredients remain in the user message.

In [14]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver stores conversation state only for this Python process.
# Use a persistent checkpointer for production applications.
agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver(),
)

### Summary: Agent Construction

`create_agent` connects the model, tools, and system prompt into a runnable graph. `InMemorySaver` enables short-term memory, but only while this notebook kernel remains alive. A production application should use a durable checkpointer when conversations must survive restarts.

In [15]:
from langchain.messages import HumanMessage

# A thread ID identifies one conversation in the checkpointer.
# Reusing this ID lets later invocations access this conversation's history.
config = {"configurable": {"thread_id": "personal-chef-demo"}}

response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="I have some leftover chicken and rice. What can I make?"
            )
        ]
    },
    config,
)

# The final message is the assistant's answer after any tool calls finish.
print(response["messages"][-1].content)

Nice! Leftover chicken and rice open up lots of tasty options. Here are some easy ideas you can straight away try (with quick notes). If you want, I can pull full recipe steps for any of these.

1) One-Pot Chicken and Rice (stovetop)
- Why try: Simple, cozy one-pan meal; uses chicken cut into bite-sized pieces and stock to cook the rice.
- Quick note: Often starts with some sautéed veggies (like carrots) and ends with a bit of butter for creaminess.
- Source vibe: Iowa Girl Eats

2) Chicken Fried Rice
- Why try: A classic use-all-the-leftovers dish; uses cooked rice, chopped chicken, peas/carrots, green onions, soy sauce, and eggs.
- Quick note: Cook the eggs first, push them aside, then fry the rice and chicken with soy sauce and sesame oil.
- Source vibe: Quick Weeknight Meals

3) Leftover Chicken and Egg Fried Rice
- Why try: Similar to fried rice, but with a simple egg-in-the-wok twist for extra protein and texture.
- Quick note: Adds frozen peas, peppers, and scallions for color a

### Summary: Invocation and Thread State

The invocation passes messages in LangChain's message format and a configuration object containing a `thread_id`. The final message is convenient for displaying the answer, while the full `response` object remains available for inspecting the tool calls and intermediate results.

In [16]:
from pprint import pprint

# Inspect the complete state to see the user message, tool calls,
# tool results, and final assistant response.
pprint(response)

{'messages': [HumanMessage(content='I have some leftover chicken and rice. What can I make?', additional_kwargs={}, response_metadata={}, id='bde398bd-90d4-44c5-9007-88b1b9a1a9d4'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 831, 'prompt_tokens': 233, 'total_tokens': 1064, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 704, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EPnWuxcDiqCsooo68Wqg58vmlb85j', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0b96c-a0b8-7fc3-9fd8-f01ef425306d-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'leftover chicken and rice recipe ideas'}, 'id': 'call_So8olIqjwTUwTsAFPHLh5EH

## Conclusion

This notebook combines a structured tool, explicit agent instructions, and thread-scoped short-term memory into a practical personal chef workflow. The same architecture can support other assistants by changing the system prompt, tool implementation, and user request while keeping the agent construction pattern intact.

For a production version, add input validation, handle search or model failures, use a durable checkpointer, and verify recipe safety details such as allergies, dietary restrictions, and food-storage guidance.